In [2]:
from transformers import logging

logging.set_verbosity_error()

In [3]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")

model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [4]:
from transformers import AutoTokenizer

prompt = "It was a dark and stormy"
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
input_ids = tokenizer(prompt).input_ids
input_ids

[1589, 436, 253, 3605, 284, 43471]

In [5]:
for t in input_ids:
    print(t, "\t:", tokenizer.decode(t))

1589 	: It
436 	:  was
253 	:  a
3605 	:  dark
284 	:  and
43471 	:  stormy


In [6]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")

In [7]:
# We tokenize again but specifying the tokenizer that we want it to
# return a PyTorch tensor, which is what the model expects,
# rather than a list of integers
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

outputs = model(input_ids)
outputs.logits.shape  # An output for each input token

torch.Size([1, 6, 49152])

In [8]:
final_logits = model(input_ids).logits[0, -1]  # The last set of logits
final_logits.argmax()  # The position of the maximum

tensor(3163)

In [9]:
tokenizer.decode(final_logits.argmax())

' night'

In [10]:
import torch

top10_logits = torch.topk(final_logits, 10)
for index in top10_logits.indices:
    print(tokenizer.decode(index))

 night
 day
 time
 evening
 winter
 sea
 morning
 month
 summer
 afternoon


In [11]:
top10 = torch.topk(final_logits.softmax(dim=0), 10)
for value, index in zip(top10.values, top10.indices):
    print(f"{tokenizer.decode(index):<10} {value.item():.2%}")

 night     50.05%
 day       17.20%
 time      5.21%
 evening   2.42%
 winter    2.19%
 sea       2.18%
 morning   1.60%
 month     1.34%
 summer    1.33%
 afternoon 1.28%


### Generating Text

In [12]:
output_ids = model.generate(input_ids, max_new_tokens=20)
decoded_text = tokenizer.decode(output_ids[0])

print("Input IDs", input_ids[0])
print("Output IDs", output_ids)
print(f"Generated text: {decoded_text}")

Input IDs tensor([ 1589,   436,   253,  3605,   284, 43471])
Output IDs tensor([[ 1589,   436,   253,  3605,   284, 43471,  3163,    28,   284,   260,
          2251,   436, 22753, 39587,    30,   198,   198,    18,  1780,   506,
          4061,    28,  1838,   582,  4817,  3413]])
Generated text: It was a dark and stormy night, and the wind was blowing fiercely.

"What's wrong, little one?" asked


In [13]:
beam_output = model.generate(
    input_ids,
    num_beams=5,
    max_new_tokens=30,
)

print(tokenizer.decode(beam_output[0]))

It was a dark and stormy night, and the sky was full of stars.

"What are those twinkling lights in the sky?" asked Timmy.

"Those are


In [14]:
beam_output = model.generate(
    input_ids,
    num_beams=5,
    repetition_penalty=2.0,
    max_new_tokens=38,
)

print(tokenizer.decode(beam_output[0]))

It was a dark and stormy night, with the sound of thunder and lightning in the distance. The sky was full of stars, and the sun was setting behind the mountains.

"What are you doing?" asked


In [15]:
from transformers import set_seed

# Setting the seed ensures we get the same results every time we run this code
set_seed(70)

sampling_output = model.generate(
    input_ids,
    do_sample=True,
    max_new_tokens=34,
    top_k=0,
)

print(tokenizer.decode(sampling_output[0]))

It was a dark and stormy time in China’s 16th century, but Cultural Revolution began shortly in the Manchurian port city of Rang in the eastern province of Heilongjiang


part added

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import torch


tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")

def generate_custom_text(prompt, temperature=1.0, topk=50, topp=0.95, max_new_tokens=40, seed=42):
    """
    Generate text using temperature scaling, top-k, and top-p sampling combined.
    """
    set_seed(seed)  # for reproducibility
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids

    output_ids = model.generate(
        input_ids,
        do_sample=True,
        temperature=temperature,
        top_k=topk,
        top_p=topp,
        max_new_tokens=max_new_tokens
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

prompt = "The future of Generative AI lies in"

text1 = generate_custom_text(prompt, temperature=0.7, topk=10, topp=0.9, max_new_tokens=40)
print(text1, "\n")

text2 = generate_custom_text(prompt, temperature=1.5, topk=5, topp=0.95, max_new_tokens=40)
print(text2)


The future of Generative AI lies in its ability to be harnessed for various applications, including education, healthcare, and creative industries. As the technology continues to evolve, it is essential to ensure that it is developed and used responsibly to benefit society 

The future of Generative AI lies in its capacity to enhance our daily lives, from automating tasks to generating personalized content for businesses and entertainment. As this technology continues to evolve, we can anticipate a future where generative AI plays an increasingly vital role


In [17]:
# Check the token IDs for the words ' positive' and ' negative'
# (note the space before the words)
tokenizer.encode(" positive"), tokenizer.encode(" negative")

([2731], [3524])

In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")

def score(review):
    """
    Zero-shot sentiment classification for SmolLM small.

    Args:
        review (str): A text review to classify.

    Returns:
        str: "Positive" or "Negative"
    """
    # Dynamically get token IDs for ' positive' and ' negative'
    positive_token_id = tokenizer.encode(" positive", add_special_tokens=False)[0]
    negative_token_id = tokenizer.encode(" negative", add_special_tokens=False)[0]

    # Adjusted prompt for better model understanding
    prompt = f"Review: {review}\nSentiment (positive/negative):"

    # Tokenize prompt
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids

    # Get logits of the last token
    final_logits = model(input_ids).logits[0, -1]

    # Convert logits to probabilities
    probs = F.softmax(final_logits, dim=0)

    # Compare probabilities
    if probs[positive_token_id] > probs[negative_token_id]:
        return "Positive"
    else:
        return "Negative"

# Test reviews
reviews = [
    "An absolutely thrilling experience from start to finish!",
    "The plot was painfully weak and the characters felt completely lifeless.",
    "Honestly, if boredom were an art form, this movie would be a masterpiece of suffering."
]

for r in reviews:
    print(score(r))


Positive
Negative
Negative
